# A/B testing for ML Model Deployment

# 1. Import Libraries


In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 2. Generate Sample Dataset

In [20]:
# For reproducibility
np.random.seed(42)

n_samples = 1000

data = pd.DataFrame({
    "feature_1": np.random.normal(50, 10, n_samples), # age
    "feature_2": np.random.normal(30, 5, n_samples),  # weight
})

# Create target column (binary classification)
data["target"] = (
    0.3 * data["feature_1"] +
    0.7 * data["feature_2"] +
    np.random.normal(0, 5, n_samples)
) > 50

data["target"] = data["target"].astype(int)

print("Sample Dataset:")
print(data.head())

Sample Dataset:
   feature_1  feature_2  target
0  54.967142  36.996777       0
1  48.617357  34.623168       0
2  56.476885  30.298152       0
3  65.230299  26.765316       0
4  47.658466  33.491117       0


# 3. Train-Test Split

In [21]:
X = data[["feature_1", "feature_2"]]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Define Candidate Models

In [22]:
prev_model_A = RandomForestClassifier(n_estimators=100)   # A 
new_model_B  = LogisticRegression() # B

models = {
    "Random Forest": prev_model_A,
    "Logistic Regression": new_model_B,  
}

results = []

# 5. Train & Evaluate Models

In [23]:
for name, model in models.items():

    # Train
    model.fit(X_train, y_train)

    # Predict
    preds = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1 Score": f1
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

print("\nModel Performance:")
print(results_df)


Model Performance:
                 Model  Accuracy  F1 Score
0        Random Forest     0.975  0.000000
1  Logistic Regression     0.980  0.333333


# 6. Decide Which Model to Deploy

In [24]:
# Industry practice: choose based on F1 or business metric
best_model_name = results_df.sort_values(
    by="F1 Score",
    ascending=False
).iloc[0]["Model"]

print("\nRecommended Model for Deployment (Based on F1 score):")
print(best_model_name)


Recommended Model for Deployment (Based on F1 score):
Logistic Regression


# 7. Mark Deployment Candidate

In [25]:

deployment_model = models[best_model_name]

print("\nModel ready for A/B Testing deployment!")


Model ready for A/B Testing deployment!
